# Лабораторная работа №2 по дисциплине "Основы нейронных сетей"

Выполнили cтуденты группы 3311:
- Шарпинский Денис
- Локтионов Тимофей
- Суздалева Алёна

Датасет: https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather?resource=download

Используемый датасет содержит почасовые данные энергосистемы Испании и погодные наблюдения по крупным городам Испании за тот же временной период. Энергетические данные описывают фактическую и прогнозную нагрузку, генерацию электроэнергии по источникам и цены. Погодные данные используются как дополнительные внешние факторы, потенциально влияющие на уровень энергопотребления. Для объединения таблиц погодные признаки агрегируются по времени, после чего соединяются с энергетическими данными по временной метке.

Можно сформулировать задачу:

По предыдущим 24 часам энергетических и погодных признаков предсказать класс нагрузки энергосистемы Испании на следующий час.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras import regularizers
from tensorflow.keras.utils import to_categorical

In [ ]:
energy_df = pd.read_csv("energy_dataset.csv")
weather_df = pd.read_csv("weather_features.csv")

In [ ]:
print("ENERGY DATASET")
print("=" * 80)

display(energy_df.head())

print("\nShape:")
print(energy_df.shape)

print("\nColumns:")
print(energy_df.columns.tolist())

print("\nInfo:")
energy_df.info()

print("\nMissing values:")
display(energy_df.isna().sum().sort_values(ascending=False).head(20))

print("\nDescribe:")
display(energy_df.describe())

In [ ]:
print("WEATHER FEATURES")
print("=" * 80)

display(weather_df.head())

print("\nShape:")
print(weather_df.shape)

print("\nColumns:")
print(weather_df.columns.tolist())

print("\nInfo:")
weather_df.info()

print("\nMissing values:")
display(weather_df.isna().sum().sort_values(ascending=False).head(20))

print("\nDescribe:")
display(weather_df.describe())

In [ ]:
print("Energy time column example:")
display(energy_df[["time"]].head())

print("Weather time column example:")
display(weather_df[["dt_iso", "city_name"]].head())

После первичного анализа данных видно, что файл `energy_dataset.csv` содержит основной почасовой временной ряд энергосистемы: фактическую нагрузку, прогноз нагрузки, генерацию по различным источникам и цены на электроэнергию. Размер датасета составляет 35064 строки и 29 столбцов, что соответствует примерно четырём годам почасовых наблюдений. В данных есть несколько полностью пустых столбцов, а также небольшое количество пропусков в отдельных энергетических признаках, поэтому потребуется очистка: удалить полностью пустые и неинформативные столбцы, а пропущенные значения заполнить или интерполировать.

Файл `weather_features.csv` содержит погодные признаки: температуру, давление, влажность, скорость ветра, осадки, облачность и другие параметры. Его размер больше, чем у энергетического датасета, потому что погодные данные представлены отдельно для нескольких городов. Поэтому напрямую объединять два файла нельзя: сначала нужно привести время к единому формату, затем агрегировать погодные данные по времени, например усреднить числовые погодные признаки по всем городам, и только после этого объединить их с энергетическими данными.

Дальнейший план работы: привести временные колонки к типу `datetime`, отсортировать данные по времени, проверить дубликаты, определить полностью пустые и константные столбцы, обработать пропуски, агрегировать погодные данные и объединить оба датасета в один многофакторный временной ряд. После этого можно будет сформировать целевую переменную, создать временные окна для RNN, нормализовать признаки и перейти к построению моделей Simple RNN, LSTM и GRU.

In [ ]:
# Копируем исходные датафреймы, чтобы не портить оригиналы
energy = energy_df.copy()
weather = weather_df.copy()

# Приводим временные колонки к datetime.
# utc=True нужен, чтобы корректно обработать часовые пояса +01:00 / +02:00.
energy["time"] = pd.to_datetime(energy["time"], utc=True)
weather["dt_iso"] = pd.to_datetime(weather["dt_iso"], utc=True)

# Сортируем по времени
energy = energy.sort_values("time").reset_index(drop=True)
weather = weather.sort_values("dt_iso").reset_index(drop=True)

print("ENERGY TIME RANGE")
print(energy["time"].min(), "->", energy["time"].max())
print()

print("WEATHER TIME RANGE")
print(weather["dt_iso"].min(), "->", weather["dt_iso"].max())
print()

print("ENERGY DUPLICATES BY TIME:")
print(energy["time"].duplicated().sum())
print()

print("WEATHER DUPLICATES BY TIME + CITY:")
print(weather.duplicated(subset=["dt_iso", "city_name"]).sum())
print()

print("WEATHER CITIES:")
print(weather["city_name"].value_counts())
print()

print("ENERGY FULLY EMPTY COLUMNS:")
empty_energy_cols = energy.columns[energy.isna().all()].tolist()
print(empty_energy_cols)
print()

print("ENERGY CONSTANT COLUMNS:")
constant_energy_cols = []
for col in energy.columns:
    if col == "time":
        continue
    if energy[col].nunique(dropna=True) <= 1:
        constant_energy_cols.append(col)

print(constant_energy_cols)
print()

print("ENERGY MISSING VALUES > 0:")
display(
    energy.isna()
    .sum()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0]
)
print()

print("WEATHER NUMERIC EXTREMES:")
weather_numeric_cols = weather.select_dtypes(include=["number"]).columns.tolist()
display(weather[weather_numeric_cols].describe().T[["min", "max", "mean", "std"]])

Временные диапазоны двух таблиц полностью совпадают: обе начинаются 2014-12-31 23:00:00 UTC и заканчиваются 2018-12-31 22:00:00 UTC. Это значит, что энергетические и погодные данные действительно относятся к одному периоду и могут быть объединены по времени.

В `energy_dataset.csv` одна строка соответствует одному часу, дубликатов по времени нет. Это хороший основной временной ряд. В нём есть два полностью пустых столбца и несколько константных столбцов, которые не несут информации для модели. Их нужно удалить. Остальные пропуски небольшие — около 17–36 значений на 35064 строки, поэтому их лучше не удалять, а заполнить интерполяцией по времени.

В `weather_features.csv` данные представлены по нескольким городам Испании. Есть 3076 дубликатов по паре `dt_iso + city_name`, а также у города ` Barcelona` заметен пробел в начале названия. Поэтому сначала нужно очистить названия городов, затем агрегировать дубликаты и только после этого усреднить погодные признаки по городам для каждого часа.

В погодных данных также видны выбросы: например, давление `1008371` и давление `0`, что физически некорректно для атмосферного давления. Такие значения нужно заменить на пропуски и затем заполнить корректными значениями. После этого погодные признаки можно объединить с энергетическими данными по временной колонке.

Дальше выполняем этап предобработки: удаляем пустые и константные энергетические признаки, заполняем пропуски, чистим погодные выбросы, агрегируем погоду по времени и объединяем две таблицы в один многофакторный временной ряд.

In [ ]:
# Создаем рабочие копии
energy_clean = energy.copy()
weather_clean = weather.copy()

# Убираем лишние пробелы в названиях городов
weather_clean["city_name"] = weather_clean["city_name"].str.strip()

# Удаляем полностью пустые и константные столбцы из energy
cols_to_drop = list(set(empty_energy_cols + constant_energy_cols))

energy_clean = energy_clean.drop(columns=cols_to_drop)

print("Удалены столбцы из energy:")
print(cols_to_drop)
print()

print("Energy shape after drop:")
print(energy_clean.shape)

In [ ]:
# Делаем time индексом, чтобы корректно интерполировать временной ряд
energy_clean = energy_clean.set_index("time")

# Интерполяция числовых пропусков по времени
energy_clean = energy_clean.interpolate(method="time")

# Если в начале/конце остались пропуски, заполняем соседними значениями
energy_clean = energy_clean.ffill().bfill()

# Возвращаем time обратно в колонку
energy_clean = energy_clean.reset_index()

print("Missing values in energy after cleaning:")
display(
    energy_clean.isna()
    .sum()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0]
)

display(energy_clean.head())

In [ ]:
# Оставляем только числовые погодные признаки, которые имеют физический смысл
weather_feature_cols = [
    "temp",
    "temp_min",
    "temp_max",
    "pressure",
    "humidity",
    "wind_speed",
    "wind_deg",
    "rain_1h",
    "rain_3h",
    "snow_3h",
    "clouds_all"
]

# Заменяем явно некорректное давление на NaN
# Нормальное атмосферное давление обычно находится примерно в диапазоне 900-1100 hPa
weather_clean.loc[
    (weather_clean["pressure"] < 900) | (weather_clean["pressure"] > 1100),
    "pressure"
] = np.nan

# Заменяем явно некорректную влажность
weather_clean.loc[
    (weather_clean["humidity"] < 0) | (weather_clean["humidity"] > 100),
    "humidity"
] = np.nan

# Заменяем явно подозрительную скорость ветра
# 133 — физически выглядит как выброс для обычного погодного датасета
weather_clean.loc[
    weather_clean["wind_speed"] > 60,
    "wind_speed"
] = np.nan

# Переводим температуру из Кельвинов в градусы Цельсия для читаемости
for col in ["temp", "temp_min", "temp_max"]:
    weather_clean[col] = weather_clean[col] - 273.15

display(weather_clean[weather_feature_cols].describe().T[["min", "max", "mean", "std"]])

In [ ]:
# 1. Сначала агрегируем дубликаты по одному городу и одному часу
weather_by_city_hour = (
    weather_clean
    .groupby(["dt_iso", "city_name"], as_index=False)[weather_feature_cols]
    .mean()
)

print("Weather shape after city-hour aggregation:")
print(weather_by_city_hour.shape)

print("Duplicates by dt_iso + city_name after aggregation:")
print(weather_by_city_hour.duplicated(subset=["dt_iso", "city_name"]).sum())
print()

# 2. Затем усредняем погодные признаки по всем городам на каждый час
weather_agg = (
    weather_by_city_hour
    .groupby("dt_iso", as_index=False)[weather_feature_cols]
    .mean()
)

# Заполняем возможные пропуски после очистки выбросов
weather_agg = weather_agg.set_index("dt_iso")
weather_agg = weather_agg.interpolate(method="time").ffill().bfill()
weather_agg = weather_agg.reset_index()

# Переименовываем колонку времени для объединения
weather_agg = weather_agg.rename(columns={"dt_iso": "time"})

print("Weather aggregated shape:")
print(weather_agg.shape)

display(weather_agg.head())

In [ ]:
df = pd.merge(
    energy_clean,
    weather_agg,
    on="time",
    how="inner"
)

print("Merged dataset shape:")
print(df.shape)

print("\nTime range:")
print(df["time"].min(), "->", df["time"].max())

print("\nMissing values after merge:")
missing_after_merge = df.isna().sum().sort_values(ascending=False)
display(missing_after_merge[missing_after_merge > 0])

display(df.head())

После очистки и объединения получен итоговый датасет размером 35064 строки и 32 столбца. Диапазон времени сохранился полностью: с 2014-12-31 23:00:00 UTC по 2018-12-31 22:00:00 UTC. Пропусков после объединения не осталось, значит энергетические и погодные данные успешно приведены к единому почасовому временному ряду.

Теперь в одной таблице находятся признаки энергосистемы Испании: генерация по источникам, прогноз и фактическая нагрузка, цены на электроэнергию, а также агрегированные погодные признаки по городам: температура, давление, влажность, ветер, осадки и облачность. Такой датасет подходит для многофакторного анализа, потому что модель будет учитывать не только прошлую нагрузку, но и дополнительные факторы, потенциально влияющие на энергопотребление.

Можно предположить 3 класса: низкая, средняя и высокая нагрузка. Необходимо просмотреть, как значения принимает нагрузка, чтобы выделить эти классы

In [ ]:
target_col = "total load actual"

print("Target column:", target_col)
print()

print("Basic statistics:")
display(df[target_col].describe())

plt.figure(figsize=(12, 4))
plt.plot(df["time"], df[target_col])
plt.title("Фактическая нагрузка энергосистемы во времени")
plt.xlabel("Время")
plt.ylabel("Total load actual")
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(df[target_col], bins=50)
plt.title("Распределение фактической нагрузки")
plt.xlabel("Total load actual")
plt.ylabel("Количество наблюдений")
plt.show()

q1 = df[target_col].quantile(0.33)
q2 = df[target_col].quantile(0.66)

print("33% quantile:", q1)
print("66% quantile:", q2)

def load_to_class(value):
    if value <= q1:
        return 0
    elif value <= q2:
        return 1
    else:
        return 2

df["load_class"] = df[target_col].apply(load_to_class)

print("\nClass distribution:")
display(df["load_class"].value_counts().sort_index())

plt.figure(figsize=(6, 4))
df["load_class"].value_counts().sort_index().plot(kind="bar")
plt.title("Распределение классов нагрузки")
plt.xlabel("Класс нагрузки")
plt.ylabel("Количество наблюдений")
plt.xticks(
    ticks=[0, 1, 2],
    labels=["Низкая", "Средняя", "Высокая"],
    rotation=0
)
plt.show()

Анализ целевой переменной `total load actual` показал, что фактическая нагрузка энергосистемы изменяется в диапазоне от 18041 до 41015. Среднее значение составляет около 28698, медиана — около 28902. По графику временного ряда видно, что данные имеют выраженную периодичность и колебания нагрузки во времени, что делает их подходящими для применения рекуррентных нейронных сетей.

Для постановки задачи многоклассовой классификации нагрузка была разделена на три класса по квантилям: низкая нагрузка — значения до 25972, средняя нагрузка — значения от 25972 до 31058, высокая нагрузка — значения выше 31058. Такой способ разбиения позволил получить почти сбалансированные классы: 11575 наблюдений низкой нагрузки, 11568 наблюдений средней нагрузки и 11921 наблюдение высокой нагрузки. Это удобно для обучения модели, так как классы представлены примерно одинаково и не требуется дополнительная балансировка.

In [ ]:
print("Columns in merged dataset:")
for i, col in enumerate(df.columns):
    print(i, col)

print("\nData types:")
display(df.dtypes)

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

print("\nNumeric columns:")
for col in numeric_cols:
    print(col)

print("\nNumber of numeric columns:", len(numeric_cols))

In [ ]:
# Добавляем календарные признаки
df["hour"] = df["time"].dt.hour
df["day_of_week"] = df["time"].dt.dayofweek
df["month"] = df["time"].dt.month

# Целевая колонка
target_col = "load_class"

# Все числовые признаки, кроме целевой переменной
feature_cols = df.select_dtypes(include=["number"]).columns.tolist()
feature_cols.remove(target_col)

print("Количество признаков:", len(feature_cols))
print("\nСписок признаков:")
for col in feature_cols:
    print("-", col)

print("\nЦелевая переменная:", target_col)

display(df[feature_cols + [target_col]].head())

In [ ]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

# Делаем отдельную рабочую копию
df_model = df.copy()

# На всякий случай сортируем по времени
df_model = df_model.sort_values("time").reset_index(drop=True)

# Параметры задачи
window_size = 24          # 24 предыдущих часа
num_classes = 3           # низкая, средняя, высокая нагрузка

# Хронологическое разделение на train/test
train_size = int(len(df_model) * 0.8)

print("Всего строк:", len(df_model))
print("Граница train/test:", train_size)
print("Train time range:", df_model.loc[0, "time"], "->", df_model.loc[train_size - 1, "time"])
print("Test time range:", df_model.loc[train_size, "time"], "->", df_model.loc[len(df_model) - 1, "time"])
print()

# Нормализуем только входные признаки.
# Важно: scaler обучаем только на train-части, чтобы не было утечки данных из test.
scaler = StandardScaler()

scaler.fit(df_model.loc[:train_size - 1, feature_cols])

features_scaled = scaler.transform(df_model[feature_cols])
targets = df_model[target_col].values

print("features_scaled shape:", features_scaled.shape)
print("targets shape:", targets.shape)

In [ ]:
def create_sequences(features, targets, window_size):
    X = []
    y = []
    target_indices = []

    for i in range(len(features) - window_size):
        # Берём 24 часа истории
        X.append(features[i:i + window_size])

        # Предсказываем класс следующего часа
        y.append(targets[i + window_size])

        # Запоминаем индекс целевой строки, чтобы потом корректно разделить train/test
        target_indices.append(i + window_size)

    return np.array(X), np.array(y), np.array(target_indices)


X, y, target_indices = create_sequences(
    features_scaled,
    targets,
    window_size
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("target_indices shape:", target_indices.shape)
print()

print("Пример:")
print("X[0] содержит строки df_model[0:24]")
print("y[0] соответствует строке df_model[24]")
print("target time:", df_model.loc[target_indices[0], "time"])
print("target class:", y[0])

In [ ]:
# Train — те последовательности, у которых целевой час находится в train-части
train_mask = target_indices < train_size

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[~train_mask]
y_test = y[~train_mask]

# One-hot encoding для categorical_crossentropy
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("y_train_cat shape:", y_train_cat.shape)
print()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("y_test_cat shape:", y_test_cat.shape)
print()

print("Train class distribution:")
print(pd.Series(y_train).value_counts().sort_index())
print()

print("Test class distribution:")
print(pd.Series(y_test).value_counts().sort_index())

Получены входные данные формы `(28027, 24, 34)` для обучения и `(7013, 24, 34)` для тестирования. Это означает, что каждый обучающий пример содержит 24 последовательных часа истории, а на каждом часе находится 34 признака. Целевая переменная — класс нагрузки следующего часа. Распределение классов в train и test выборках примерно сбалансировано, поэтому на этом этапе дополнительная балансировка классов не требуется.

In [ ]:
### ЭТО ФУНКЦИЯ УТИЛИТА, ЧТОБЫ ДЕЛАТЬ КРАСИВЫЙ ВЫВОД ГРАФИКОВ ПОСЛЕ ОБУЧЕНИЯ МОДЕЛЕЙ.

import matplotlib.pyplot as plt
import pandas as pd

def plot_model_results(results_df, model_name):
    df_plot = results_df.copy()

    # Для красивых подписей
    df_plot["label"] = (
        df_plot["optimizer"].astype(str)
        + " + "
        + df_plot["regularization"].astype(str)
    )

    # Сортируем по accuracy по убыванию
    df_plot = df_plot.sort_values(by="test_accuracy", ascending=False).reset_index(drop=True)

    fig, axes = plt.subplots(2, 1, figsize=(12, 10))

    # ---------- Accuracy ----------
    axes[0].barh(df_plot["label"], df_plot["test_accuracy"])
    axes[0].set_title(f"Сравнение конфигураций {model_name} — Accuracy")
    axes[0].set_xlabel("Test accuracy")
    axes[0].set_ylabel("Конфигурация")
    axes[0].invert_yaxis()

    # Подписи значений
    for i, v in enumerate(df_plot["test_accuracy"]):
        axes[0].text(v + 0.001, i, f"{v:.4f}", va="center")

    # ---------- Loss ----------
    axes[1].barh(df_plot["label"], df_plot["test_loss"])
    axes[1].set_title(f"Сравнение конфигураций {model_name} — Loss")
    axes[1].set_xlabel("Test loss")
    axes[1].set_ylabel("Конфигурация")
    axes[1].invert_yaxis()

    # Подписи значений
    for i, v in enumerate(df_plot["test_loss"]):
        axes[1].text(v + 0.001, i, f"{v:.4f}", va="center")

    plt.tight_layout()
    plt.show()

### Simple RNN

Можно приступать к созданию простой рнн.

Для каждой архитектуры:
SimpleRNN
LSTM
GRU

проверить:
SGD
Momentum
RMSProp
Adam

и регуляризацию:
без регуляризации
L1
L2

### Памятка про окна.

На случай, если я забуду, почему используются окна:

вся задача сводится к тому, чтобы на основе предыдущих 24 часов выдать предсказание 25ого часа. в таком случае рнн вычисляет хидден стейт для всего 24 инпутов из 34 векторов и потом пара денс слоев. и софтмакс ну и да, небольшой такой момент, что мы вот эту операцию, вышеописанную делаем (28000 - 23) раз (для трейн).

X[i] = 24 часа истории
     = матрица 24 × 34

y[i] = класс нагрузки следующего часа


24 вектора по 34 признака
↓
SimpleRNN считает h₁, h₂, ..., h₂₄
↓
берём последнее состояние h₂₄
↓
Dense(32)
↓
Dense(3)
↓
Softmax
↓
вероятности трёх классов нагрузки 25-го часа


35064 исходных часа
window_size = 24

35064 - 24 = 35040 окон

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Dropout

def get_optimizer(optimizer_name):
    if optimizer_name == "sgd":
        return SGD(learning_rate=0.01)
    
    if optimizer_name == "momentum":
        return SGD(learning_rate=0.01, momentum=0.9)
    
    if optimizer_name == "rmsprop":
        return RMSprop(learning_rate=0.001)
    
    if optimizer_name == "adam":
        return Adam(learning_rate=0.001)
    
    raise ValueError(f"Unknown optimizer: {optimizer_name}")


def get_regularizer(reg_type, strength=0.001):
    if reg_type is None:
        return None
    
    if reg_type == "l1":
        return regularizers.l1(strength)
    
    if reg_type == "l2":
        return regularizers.l2(strength)
    
    raise ValueError(f"Unknown regularization: {reg_type}")


def build_simple_rnn_model(
    input_shape,
    num_classes,
    optimizer_name="adam",
    reg_type=None,
    reg_strength=0.001,
    units=64,
    dense_units=32,
    dropout_rate=0.2
):
    reg = get_regularizer(reg_type, reg_strength)
    optimizer = get_optimizer(optimizer_name)

    model = Sequential([
        Input(shape=input_shape),

        SimpleRNN(
            units=units,
            activation="tanh",
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            bias_regularizer=reg,
            name="simple_rnn_layer"
        ),

        Dropout(dropout_rate),

        Dense(
            units=dense_units,
            activation="relu",
            kernel_regularizer=reg,
            bias_regularizer=reg,
            name="dense_hidden"
        ),

        Dense(
            units=num_classes,
            activation="softmax",
            name="output_layer"
        )
    ])

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
simple_rnn_experiments = []

optimizers_to_test = ["sgd", "momentum", "rmsprop", "adam"]
regularizations_to_test = [None, "l1", "l2"]

for optimizer_name in optimizers_to_test:
    for reg_type in regularizations_to_test:
        simple_rnn_experiments.append({
            "architecture": "SimpleRNN",
            "optimizer": optimizer_name,
            "regularization": reg_type
        })

simple_rnn_experiments

In [ ]:
simple_rnn_results = []
simple_rnn_histories = {}
simple_rnn_models = {}

for exp in simple_rnn_experiments:
    optimizer_name = exp["optimizer"]
    reg_type = exp["regularization"]

    exp_name = f"SimpleRNN_{optimizer_name}_{reg_type if reg_type else 'no_reg'}"
    print("=" * 80)
    print("Training:", exp_name)
    print("=" * 80)

    np.random.seed(42)
    tf.random.set_seed(42)

    model = build_simple_rnn_model(
        input_shape=input_shape,
        num_classes=num_classes,
        optimizer_name=optimizer_name,
        reg_type=reg_type,
        reg_strength=0.001,
        units=64,
        dense_units=32,
        dropout_rate=0.2
    )

    # Новый EarlyStopping для каждой модели
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train_cat,
        validation_split=0.2,
        epochs=15,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(
        X_test,
        y_test_cat,
        verbose=0
    )

    simple_rnn_results.append({
        "architecture": "SimpleRNN",
        "optimizer": optimizer_name,
        "regularization": reg_type if reg_type else "none",
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "epochs_trained": len(history.history["loss"])
    })

    simple_rnn_histories[exp_name] = history
    simple_rnn_models[exp_name] = model

    print(f"{exp_name} test loss: {test_loss:.4f}")
    print(f"{exp_name} test accuracy: {test_accuracy:.4f}")

In [ ]:
simple_rnn_results_df = pd.DataFrame(simple_rnn_results)
simple_rnn_results_df = simple_rnn_results_df.sort_values(
    by="test_accuracy",
    ascending=False
)

display(simple_rnn_results_df)

In [ ]:
plot_model_results(simple_rnn_results_df, "SimpleRNN")

In [ ]:
best_simple_rnn_row = simple_rnn_results_df.iloc[0]

best_simple_rnn_name = (
    f"SimpleRNN_{best_simple_rnn_row['optimizer']}_"
    f"{best_simple_rnn_row['regularization'] if best_simple_rnn_row['regularization'] != 'none' else 'no_reg'}"
)

print("Best SimpleRNN model:")
print(best_simple_rnn_name)

best_simple_rnn_model = simple_rnn_models[best_simple_rnn_name]
best_simple_rnn_history = simple_rnn_histories[best_simple_rnn_name]

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(best_simple_rnn_history.history["accuracy"], label="train")
plt.plot(best_simple_rnn_history.history["val_accuracy"], label="validation")
plt.title(f"{best_simple_rnn_name} accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(best_simple_rnn_history.history["loss"], label="train")
plt.plot(best_simple_rnn_history.history["val_loss"], label="validation")
plt.title(f"{best_simple_rnn_name} loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

### LSTM

In [ ]:
from tensorflow.keras.layers import LSTM

def build_lstm_model(
    input_shape,
    num_classes,
    optimizer_name="adam",
    reg_type=None,
    reg_strength=0.001,
    units=64,
    dense_units=32,
    dropout_rate=0.2
):
    reg = get_regularizer(reg_type, reg_strength)
    optimizer = get_optimizer(optimizer_name)

    model = Sequential([
        Input(shape=input_shape),

        LSTM(
            units=units,
            activation="tanh",
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            bias_regularizer=reg,
            name="lstm_layer"
        ),

        Dropout(dropout_rate),

        Dense(
            units=dense_units,
            activation="relu",
            kernel_regularizer=reg,
            bias_regularizer=reg,
            name="dense_hidden"
        ),

        Dense(
            units=num_classes,
            activation="softmax",
            name="output_layer"
        )
    ])

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
lstm_experiments = []

optimizers_to_test = ["sgd", "momentum", "rmsprop", "adam"]
regularizations_to_test = [None, "l1", "l2"]

for optimizer_name in optimizers_to_test:
    for reg_type in regularizations_to_test:
        lstm_experiments.append({
            "architecture": "LSTM",
            "optimizer": optimizer_name,
            "regularization": reg_type
        })

lstm_experiments

In [ ]:
lstm_results = []
lstm_histories = {}
lstm_models = {}

for exp in lstm_experiments:
    optimizer_name = exp["optimizer"]
    reg_type = exp["regularization"]

    exp_name = f"LSTM_{optimizer_name}_{reg_type if reg_type else 'no_reg'}"
    print("=" * 80)
    print("Training:", exp_name)
    print("=" * 80)

    np.random.seed(42)
    tf.random.set_seed(42)

    model = build_lstm_model(
        input_shape=input_shape,
        num_classes=num_classes,
        optimizer_name=optimizer_name,
        reg_type=reg_type,
        reg_strength=0.001,
        units=64,
        dense_units=32,
        dropout_rate=0.2
    )

    # Новый EarlyStopping для каждой модели
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train_cat,
        validation_split=0.2,
        epochs=15,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(
        X_test,
        y_test_cat,
        verbose=0
    )

    lstm_results.append({
        "architecture": "LSTM",
        "optimizer": optimizer_name,
        "regularization": reg_type if reg_type else "none",
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "epochs_trained": len(history.history["loss"])
    })

    lstm_histories[exp_name] = history
    lstm_models[exp_name] = model

    print(f"{exp_name} test loss: {test_loss:.4f}")
    print(f"{exp_name} test accuracy: {test_accuracy:.4f}")

In [ ]:
lstm_results_df = pd.DataFrame(lstm_results)
lstm_results_df = lstm_results_df.sort_values(
    by="test_accuracy",
    ascending=False
)

display(lstm_results_df)

In [ ]:
plot_model_results(lstm_results_df, "LSTM")

In [ ]:
best_lstm_row = lstm_results_df.iloc[0]

best_lstm_name = (
    f"LSTM_{best_lstm_row['optimizer']}_"
    f"{best_lstm_row['regularization'] if best_lstm_row['regularization'] != 'none' else 'no_reg'}"
)

print("Best LSTM model:")
print(best_lstm_name)

best_lstm_model = lstm_models[best_lstm_name]
best_lstm_history = lstm_histories[best_lstm_name]

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(best_lstm_history.history["accuracy"], label="train")
plt.plot(best_lstm_history.history["val_accuracy"], label="validation")
plt.title(f"{best_lstm_name} accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(best_lstm_history.history["loss"], label="train")
plt.plot(best_lstm_history.history["val_loss"], label="validation")
plt.title(f"{best_lstm_name} loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

### GRU

In [ ]:
from tensorflow.keras.layers import GRU

def build_gru_model(
    input_shape,
    num_classes,
    optimizer_name="adam",
    reg_type=None,
    reg_strength=0.001,
    units=64,
    dense_units=32,
    dropout_rate=0.2
):
    reg = get_regularizer(reg_type, reg_strength)
    optimizer = get_optimizer(optimizer_name)

    model = Sequential([
        Input(shape=input_shape),

        GRU(
            units=units,
            activation="tanh",
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            bias_regularizer=reg,
            name="gru_layer"
        ),

        Dropout(dropout_rate),

        Dense(
            units=dense_units,
            activation="relu",
            kernel_regularizer=reg,
            bias_regularizer=reg,
            name="dense_hidden"
        ),

        Dense(
            units=num_classes,
            activation="softmax",
            name="output_layer"
        )
    ])

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
gru_experiments = []

optimizers_to_test = ["sgd", "momentum", "rmsprop", "adam"]
regularizations_to_test = [None, "l1", "l2"]

for optimizer_name in optimizers_to_test:
    for reg_type in regularizations_to_test:
        gru_experiments.append({
            "architecture": "GRU",
            "optimizer": optimizer_name,
            "regularization": reg_type
        })

gru_experiments

In [ ]:
gru_results = []
gru_histories = {}
gru_models = {}

for exp in gru_experiments:
    optimizer_name = exp["optimizer"]
    reg_type = exp["regularization"]

    exp_name = f"GRU_{optimizer_name}_{reg_type if reg_type else 'no_reg'}"
    print("=" * 80)
    print("Training:", exp_name)
    print("=" * 80)

    np.random.seed(42)
    tf.random.set_seed(42)

    model = build_gru_model(
        input_shape=input_shape,
        num_classes=num_classes,
        optimizer_name=optimizer_name,
        reg_type=reg_type,
        reg_strength=0.001,
        units=64,
        dense_units=32,
        dropout_rate=0.2
    )

    # ВАЖНО: новый EarlyStopping для каждой модели
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train_cat,
        validation_split=0.2,
        epochs=15,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(
        X_test,
        y_test_cat,
        verbose=0
    )

    gru_results.append({
        "architecture": "GRU",
        "optimizer": optimizer_name,
        "regularization": reg_type if reg_type else "none",
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "epochs_trained": len(history.history["loss"])
    })

    gru_histories[exp_name] = history
    gru_models[exp_name] = model

    print(f"{exp_name} test loss: {test_loss:.4f}")
    print(f"{exp_name} test accuracy: {test_accuracy:.4f}")

In [ ]:
gru_results_df = pd.DataFrame(gru_results)
gru_results_df = gru_results_df.sort_values(
    by="test_accuracy",
    ascending=False
)

display(gru_results_df)

In [ ]:
plot_model_results(gru_results_df, "GRU")

In [ ]:
best_gru_row = gru_results_df.iloc[0]

best_gru_name = (
    f"GRU_{best_gru_row['optimizer']}_"
    f"{best_gru_row['regularization'] if best_gru_row['regularization'] != 'none' else 'no_reg'}"
)

print("Best GRU model:")
print(best_gru_name)

best_gru_model = gru_models[best_gru_name]
best_gru_history = gru_histories[best_gru_name]

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(best_gru_history.history["accuracy"], label="train")
plt.plot(best_gru_history.history["val_accuracy"], label="validation")
plt.title(f"{best_gru_name} accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(best_gru_history.history["loss"], label="train")
plt.plot(best_gru_history.history["val_loss"], label="validation")
plt.title(f"{best_gru_name} loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

### Лучшие и худшие модели

In [ ]:
def get_experiment_name(architecture, optimizer, regularization):
    reg_part = regularization if regularization != "none" else "no_reg"
    return f"{architecture}_{optimizer}_{reg_part}"


def plot_history(history, title):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="train")
    plt.plot(history.history["val_accuracy"], label="validation")
    plt.title(f"{title} accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="validation")
    plt.title(f"{title} loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()


def show_best_and_worst(results_df, histories, architecture_name):
    df_sorted = results_df.sort_values(by="test_accuracy", ascending=False).reset_index(drop=True)

    best_row = df_sorted.iloc[0]
    worst_row = df_sorted.iloc[-1]

    summary_df = pd.DataFrame([
        {
            "type": "best",
            "architecture": architecture_name,
            "optimizer": best_row["optimizer"],
            "regularization": best_row["regularization"],
            "test_accuracy": best_row["test_accuracy"],
            "test_loss": best_row["test_loss"],
            "epochs_trained": best_row["epochs_trained"],
        },
        {
            "type": "worst",
            "architecture": architecture_name,
            "optimizer": worst_row["optimizer"],
            "regularization": worst_row["regularization"],
            "test_accuracy": worst_row["test_accuracy"],
            "test_loss": worst_row["test_loss"],
            "epochs_trained": worst_row["epochs_trained"],
        }
    ])

    print("=" * 80)
    print(f"{architecture_name}: лучшая и худшая конфигурации")
    print("=" * 80)
    display(summary_df)

    best_name = get_experiment_name(
        architecture_name,
        best_row["optimizer"],
        best_row["regularization"]
    )

    worst_name = get_experiment_name(
        architecture_name,
        worst_row["optimizer"],
        worst_row["regularization"]
    )

    print("Best model name:", best_name)
    print("Worst model name:", worst_name)

    print("\nГрафики лучшей модели:")
    plot_history(histories[best_name], f"{best_name} — best")

    print("\nГрафики худшей модели:")
    plot_history(histories[worst_name], f"{worst_name} — worst")

    return summary_df


simple_rnn_best_worst = show_best_and_worst(
    simple_rnn_results_df,
    simple_rnn_histories,
    "SimpleRNN"
)

lstm_best_worst = show_best_and_worst(
    lstm_results_df,
    lstm_histories,
    "LSTM"
)

gru_best_worst = show_best_and_worst(
    gru_results_df,
    gru_histories,
    "GRU"
)